In [ ]:
import pandas as pd
from google.colab import files

In [ ]:
skills_df = pd.read_excel("Skills.xlsx")
content_df = pd.read_excel("Content Model Reference.xlsx")
occupation_df = pd.read_excel("occupation_data_clean.xlsx")

In [ ]:
print("Skills columns:", skills_df.columns.tolist())
print("Content columns:", content_df.columns.tolist())
print("Occupation columns:", occupation_df.columns.tolist())

Skills columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']
Content columns: ['Element ID', 'Element Name', 'Description']
Occupation columns: ['O*NET-SOC Code', 'Title', 'Description']


**Keeping only IM and LV (in Scale ID) rows from Skills**

In [ ]:
skills_filtered = skills_df[skills_df["Scale ID"].isin(["IM", "LV"])].copy()
skills_filtered.head()

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.1830,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst


In [ ]:
skills_filtered = skills_filtered[[
    "O*NET-SOC Code",
    "Title",
    "Element ID",
    "Element Name",
    "Scale ID",
    "Data Value"
]]

In [ ]:
skills_pivot = skills_filtered.pivot_table(
    index=["O*NET-SOC Code", "Title", "Element ID", "Element Name"],
    columns="Scale ID",
    values="Data Value",
    aggfunc="first"
).reset_index()

In [ ]:
skills_pivot = skills_pivot.rename(columns={
    "O*NET-SOC Code": "soc_code",
    "Title": "occupation_title",
    "Element ID": "skill_id",
    "Element Name": "skill_name",
    "IM": "importance",
    "LV": "level"
})

In [ ]:
skills_pivot.head()

Scale ID,soc_code,occupation_title,skill_id,skill_name,importance,level
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,4.12,4.62
1,11-1011.00,Chief Executives,2.A.1.b,Active Listening,4.00,4.75
2,11-1011.00,Chief Executives,2.A.1.c,Writing,4.12,4.38
3,11-1011.00,Chief Executives,2.A.1.d,Speaking,4.25,4.75
4,11-1011.00,Chief Executives,2.A.1.e,Mathematics,3.25,3.50


In [ ]:
content_clean = content_df[["Element ID", "Element Name", "Description"]].copy()
content_clean = content_clean.rename(columns={
    "Element ID": "skill_id",
    "Element Name": "content_skill_name",
    "Description": "skill_description"
})
content_clean.head()

,skill_id,content_skill_name,skill_description
0,1,Worker Characteristics,Worker Characteristics
1,1.A,Abilities,Enduring attributes of the individual that inf...
2,1.A.1,Cognitive Abilities,Abilities that influence the acquisition and a...
3,1.A.1.a,Verbal Abilities,Abilities that influence the acquisition and a...
4,1.A.1.a.1,Oral Comprehension,The ability to listen to and understand inform...


In [ ]:
skills_enriched = skills_pivot.merge(
    content_clean,
    on="skill_id",
    how="left"
)
skills_enriched.head()

,soc_code,occupation_title,skill_id,skill_name,importance,level,content_skill_name,skill_description
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,4.12,4.62,Reading Comprehension,Understanding written sentences and paragraphs...
1,11-1011.00,Chief Executives,2.A.1.b,Active Listening,4.00,4.75,Active Listening,Giving full attention to what other people are...
2,11-1011.00,Chief Executives,2.A.1.c,Writing,4.12,4.38,Writing,Communicating effectively in writing as approp...
3,11-1011.00,Chief Executives,2.A.1.d,Speaking,4.25,4.75,Speaking,Talking to others to convey information effect...
4,11-1011.00,Chief Executives,2.A.1.e,Mathematics,3.25,3.50,Mathematics,Using mathematics to solve problems.


In [ ]:
occupation_df.head()
print(occupation_df.columns.tolist())

['O*NET-SOC Code', 'Title', 'Description']


In [ ]:
occupation_clean = occupation_df.rename(columns={
    "O*NET-SOC Code": "soc_code",
    "Title": "occupation_title_clean",
    "Description": "occupation_description"
})

In [ ]:
occupation_clean = occupation_clean[[
    "soc_code",
    "occupation_title_clean",
    "occupation_description"
]]

In [ ]:
final_kg_table = skills_enriched.merge(
    occupation_clean,
    on="soc_code",
    how="left"
)

In [ ]:
final_kg_table = final_kg_table[[
    "soc_code",
    "occupation_title_clean",
    "occupation_description",
    "skill_id",
    "skill_name",
    "content_skill_name",
    "skill_description",
    "importance",
    "level"
]]

In [ ]:
final_kg_table = final_kg_table.rename(columns={
    "occupation_title_clean": "occupation_title"
})

In [ ]:
print(final_kg_table.shape)
print(final_kg_table.isnull().sum())
final_kg_table.head(10)

(31290, 9)
soc_code                  0
occupation_title          0
occupation_description    0
skill_id                  0
skill_name                0
content_skill_name        0
skill_description         0
importance                0
level                     0
dtype: int64


,soc_code,occupation_title,occupation_description,skill_id,skill_name,content_skill_name,skill_description,importance,level
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.b,Active Listening,Active Listening,Giving full attention to what other people are...,4.00,4.75
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.c,Writing,Writing,Communicating effectively in writing as approp...,4.12,4.38
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.d,Speaking,Speaking,Talking to others to convey information effect...,4.25,4.75
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.e,Mathematics,Mathematics,Using mathematics to solve problems.,3.25,3.50
5,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.f,Science,Science,Using scientific rules and methods to solve pr...,1.62,0.75
6,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.2.a,Critical Thinking,Critical Thinking,Using logic and reasoning to identify the stre...,4.38,4.75
7,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.2.b,Active Learning,Active Learning,Understanding the implications of new informat...,3.75,4.50
8,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.2.c,Learning Strategies,Learning Strategies,Selecting and using training/instructional met...,3.12,3.75
9,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.2.d,Monitoring,Monitoring,"Monitoring/Assessing performance of yourself, ...",4.00,5.25


In [ ]:
duplicates = final_kg_table.duplicated(subset=["soc_code", "skill_id"]).sum()
print("Duplicate occupation-skill pairs:", duplicates)

Duplicate occupation-skill pairs: 0


In [ ]:
final_kg_table.to_csv("final_kg_table.csv", index=False)
final_kg_table.to_excel("final_kg_table.xlsx", index=False)

## **Generating ONET occupation and skill nodes and edges csv's**


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("onet_kg_table.csv")

print("Shape:", df.shape)
df.head()

Shape: (31290, 9)


,soc_code,occupation_title,occupation_description,skill_id,skill_name,content_skill_name,skill_description,importance,level
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.b,Active Listening,Active Listening,Giving full attention to what other people are...,4.00,4.75
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.c,Writing,Writing,Communicating effectively in writing as approp...,4.12,4.38
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.d,Speaking,Speaking,Talking to others to convey information effect...,4.25,4.75
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.e,Mathematics,Mathematics,Using mathematics to solve problems.,3.25,3.50


In [ ]:
print(df.columns.tolist())

['soc_code', 'occupation_title', 'occupation_description', 'skill_id', 'skill_name', 'content_skill_name', 'skill_description', 'importance', 'level']


###**Generate onet_occupations.csv**

In [ ]:
onet_occupations = df[
    ['soc_code', 'occupation_title', 'occupation_description']
].drop_duplicates()

onet_occupations = onet_occupations.rename(columns={
    'occupation_title': 'title',
    'occupation_description': 'description'
})

print("Occupations:", onet_occupations.shape)
onet_occupations.head()

Occupations: (894, 3)


,soc_code,title,description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
35,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
70,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
105,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
140,11-2021.00,Marketing Managers,"Plan, direct, or coordinate marketing policies..."


### **Generate onet_skills.csv**

In [ ]:
onet_skills = df[
    ['skill_id', 'skill_name', 'skill_description']
].drop_duplicates()

onet_skills = onet_skills.rename(columns={
    'skill_name': 'name',
    'skill_description': 'description'
})

print("Skills:", onet_skills.shape)
onet_skills.head()

Skills: (35, 3)


,skill_id,name,description
0,2.A.1.a,Reading Comprehension,Understanding written sentences and paragraphs...
1,2.A.1.b,Active Listening,Giving full attention to what other people are...
2,2.A.1.c,Writing,Communicating effectively in writing as approp...
3,2.A.1.d,Speaking,Talking to others to convey information effect...
4,2.A.1.e,Mathematics,Using mathematics to solve problems.


### **Generate onet_requires_skill.csv**

In [ ]:
onet_requires_skill = df[
    ['soc_code', 'skill_id', 'importance', 'level']
].drop_duplicates()

In [ ]:
onet_requires_skill['importance'] = pd.to_numeric(
    onet_requires_skill['importance'], errors='coerce'
)

onet_requires_skill['level'] = pd.to_numeric(
    onet_requires_skill['level'], errors='coerce'
)

In [ ]:
onet_requires_skill = onet_requires_skill.dropna(
    subset=['importance', 'level']
)

In [ ]:
onet_requires_skill = onet_requires_skill.drop_duplicates(
    subset=['soc_code', 'skill_id']
)

In [ ]:
print("Edges:", onet_requires_skill.shape)
onet_requires_skill.head()

Edges: (31290, 4)


,soc_code,skill_id,importance,level
0,11-1011.00,2.A.1.a,4.12,4.62
1,11-1011.00,2.A.1.b,4.00,4.75
2,11-1011.00,2.A.1.c,4.12,4.38
3,11-1011.00,2.A.1.d,4.25,4.75
4,11-1011.00,2.A.1.e,3.25,3.50


### **Saving csv files**

In [ ]:
onet_occupations.to_csv("onet_occupations.csv", index=False)
onet_skills.to_csv("onet_skills.csv", index=False)
onet_requires_skill.to_csv("onet_requires_skill.csv", index=False)

## **Generating crosswalk csv file**

In [ ]:
crosswalk_df = pd.read_csv("onet_with_esco.csv")

print("Shape:", crosswalk_df.shape)
crosswalk_df.head()

Shape: (139090, 12)


,soc_code,occupation_title,occupation_description,skill_id,skill_name,content_skill_name,skill_description,importance,level,esco_occ_uri,esco_occ_label,onet_esco_match_type
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,closeMatch
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,exactMatch
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,broadMatch
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,broadMatch
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,2.A.1.a,Reading Comprehension,Reading Comprehension,Understanding written sentences and paragraphs...,4.12,4.62,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,broadMatch


In [ ]:
print(crosswalk_df.columns.tolist())

['soc_code', 'occupation_title', 'occupation_description', 'skill_id', 'skill_name', 'content_skill_name', 'skill_description', 'importance', 'level', 'esco_occ_uri', 'esco_occ_label', 'onet_esco_match_type']


### **Extract Occupation-Level Crosswalk**  
We only keep occupation mapping columns:

In [ ]:
onet_esco_crosswalk = crosswalk_df[
    [
        "soc_code",
        "occupation_title",
        "esco_occ_uri",
        "esco_occ_label",
        "onet_esco_match_type"
    ]
]

### **Remove duplicates**  
Because ESCO mapping repeats for each skill row:

In [ ]:
onet_esco_crosswalk = onet_esco_crosswalk.drop_duplicates(
    subset=["soc_code", "esco_occ_uri", "onet_esco_match_type"]
)

### **Remove null values**

In [ ]:
onet_esco_crosswalk = onet_esco_crosswalk.dropna(
    subset=["soc_code", "esco_occ_uri"]
)

### **Rename columns (cleaner Neo4j usage)**

In [ ]:
onet_esco_crosswalk = onet_esco_crosswalk.rename(columns={
    "occupation_title": "onet_occ_title",
    "esco_occ_label": "esco_occ_title",
    "onet_esco_match_type": "match_type"
})

### **Checking output**

In [ ]:
print("Crosswalk shape:", onet_esco_crosswalk.shape)
onet_esco_crosswalk.head()

Crosswalk shape: (3955, 5)


,soc_code,onet_occ_title,esco_occ_uri,esco_occ_title,match_type
0,11-1011.00,Chief Executives,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,closeMatch
1,11-1011.00,Chief Executives,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,exactMatch
2,11-1011.00,Chief Executives,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,broadMatch
3,11-1011.00,Chief Executives,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,broadMatch
4,11-1011.00,Chief Executives,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,broadMatch


### **Save to csv file**

In [ ]:
onet_esco_crosswalk.to_csv(
    "onet_esco_occ_crosswalk.csv",
    index=False
)

In [ ]:
onet_esco_crosswalk["match_type"].value_counts()

,count
match_type,
broadMatch,1871
closeMatch,1383
exactMatch,483
narrowMatch,218


### **Creating a filtered version for high confidence matches only from crosswalk.csv file**

In [ ]:
high_conf_crosswalk = onet_esco_crosswalk[
    onet_esco_crosswalk["match_type"].isin(["exactMatch", "closeMatch"])
].copy()

print(high_conf_crosswalk["match_type"].value_counts())
print(high_conf_crosswalk.shape)

high_conf_crosswalk.to_csv("onet_esco_occ_crosswalk_high_conf.csv", index=False)

match_type
closeMatch    1383
exactMatch     483
Name: count, dtype: int64
(1866, 5)


# **Generate clean ESCO files**

### **ESCO occupations**

In [ ]:
import pandas as pd

occ = pd.read_csv("occupations_en.csv")

esco_occupations = occ[[
    "conceptUri",
    "preferredLabel",
    "description",
    "iscoGroup",
    "code"
]].drop_duplicates()

esco_occupations.columns = [
    "esco_occ_uri",
    "esco_occ_label",
    "description",
    "isco_group",
    "code"
]

esco_occupations.to_csv("esco_occupations.csv", index=False)

print("ESCO occupations created:", esco_occupations.shape)

ESCO occupations created: (3039, 5)


### **ESCO skills**

In [ ]:
skills = pd.read_csv("skills_en.csv")

esco_skills = skills[[
    "conceptUri",
    "preferredLabel",
    "skillType",
    "description"
]].drop_duplicates()

esco_skills.columns = [
    "esco_skill_uri",
    "esco_skill_label",
    "skill_type",
    "description"
]

esco_skills.to_csv("esco_skills.csv", index=False)

print("ESCO skills created:", esco_skills.shape)

ESCO skills created: (13939, 4)


### **ESCO occupation-skill edges**

In [ ]:
occ_skill = pd.read_csv("occupationSkillRelations_en.csv")

esco_occ_skill = occ_skill[[
    "occupationUri",
    "skillUri",
    "relationType",
    "skillType"
]].drop_duplicates()

esco_occ_skill.columns = [
    "esco_occ_uri",
    "esco_skill_uri",
    "relation_type",
    "skill_type"
]

esco_occ_skill.to_csv("esco_occ_requires_skill.csv", index=False)

print("ESCO occupation-skill edges:", esco_occ_skill.shape)

ESCO occupation-skill edges: (126051, 4)


### **Skill hierarchy**

In [ ]:
hier = pd.read_csv("skillsHierarchy_en.csv")

esco_skill_hierarchy = hier[[
    "Level 0 URI",
    "Level 1 URI"
]].dropna().drop_duplicates()

esco_skill_hierarchy.columns = [
    "parent_skill_uri",
    "child_skill_uri"
]

esco_skill_hierarchy.to_csv("esco_skill_hierarchy.csv", index=False)

print("Skill hierarchy created:", esco_skill_hierarchy.shape)

Skill hierarchy created: (28, 2)


In [ ]:
import pandas as pd

hier = pd.read_csv("esco_skill_hierarchy.csv")
print(hier.shape)
print(hier.columns.tolist())
hier.head()

(28, 2)
['parent_skill_uri', 'child_skill_uri']


,parent_skill_uri,child_skill_uri
0,http://data.europa.eu/esco/skill/e35a5936-091d...,http://data.europa.eu/esco/skill/43f425aa-f45d...
1,http://data.europa.eu/esco/skill/e35a5936-091d...,http://data.europa.eu/esco/skill/e434e71a-f068...
2,http://data.europa.eu/esco/skill/335228d2-297d...,http://data.europa.eu/esco/skill/03e0b95b-67d1...
3,http://data.europa.eu/esco/skill/335228d2-297d...,http://data.europa.eu/esco/skill/0a2d70ee-d435...
4,http://data.europa.eu/esco/skill/335228d2-297d...,http://data.europa.eu/esco/skill/243eb885-07c7...


# **Cleaning csv files for TECH roles only**

In [ ]:
import pandas as pd
import re
from collections import Counter

# ----------------------------
# 1) Load files
# ----------------------------
onet = pd.read_csv('/content/onet_occupations.csv')
esco = pd.read_csv('/content/esco_occupations.csv')
crosswalk = pd.read_csv('/content/onet_esco_occ_crosswalk.csv')

# ----------------------------
# 2) Helper: find column names safely
# ----------------------------
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}")

onet_title_col = pick_col(onet, ['title'])
esco_title_col = pick_col(esco, ['esco_occ_label', 'label', 'esco_occ_title'])

# ----------------------------
# 3) Normalize text
# ----------------------------
def norm(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ----------------------------
# 4) Tech keyword patterns
#    Use these first to inspect matches.
# ----------------------------
keyword_patterns = {
    "software engineer": r"\bsoftware engineer(?:s)?\b",
    "software developer": r"\bsoftware developer(?:s)?\b",
    "developer": r"\bdeveloper(?:s)?\b",
    "programmer": r"\bprogrammer(?:s)?\b",
    "computer": r"\bcomputer\b",
    "information systems": r"\binformation systems?\b",
    "information technology": r"\binformation technology\b",
    "ICT": r"\bict\b",
    "data scientist": r"\bdata scientist(?:s)?\b",
    "data engineer": r"\bdata engineer(?:s)?\b",
    "data analyst": r"\bdata analyst(?:s)?\b",
    "machine learning": r"\bmachine learning\b",
    "artificial intelligence": r"\bartificial intelligence\b",
    "cloud": r"\bcloud\b",
    "devops": r"\bdevops\b",
    "security": r"\bsecurity\b",
    "cyber": r"\bcyber\b",
    "network": r"\bnetwork\b",
    "database": r"\bdatabase\b",
    "system administrator": r"\bsystems? administrator\b",
    "systems analyst": r"\bsystems analyst\b",
    "quality assurance": r"\bquality assurance\b",
    "tester": r"\btester(?:s)?\b|\btesting\b",
    "architect": r"\barchitect(?:s)?\b",
}

def get_matches(text):
    text = norm(text)
    return [kw for kw, pat in keyword_patterns.items() if re.search(pat, text)]

def add_keyword_matches(df, title_col):
    out = df.copy()
    out['title_norm'] = out[title_col].apply(norm)
    out['matched_keywords'] = out['title_norm'].apply(get_matches)
    out['matched_keywords_str'] = out['matched_keywords'].apply(lambda xs: '; '.join(xs))
    return out

onet_review = add_keyword_matches(onet, onet_title_col)
esco_review = add_keyword_matches(esco, esco_title_col)

# ----------------------------
# 5) See which keyword matches exist
# ----------------------------
def keyword_counts(df):
    c = Counter(k for hits in df['matched_keywords'] for k in hits)
    return pd.DataFrame(c.items(), columns=['keyword', 'count']).sort_values('count', ascending=False)

print("ONET keyword counts")
display(keyword_counts(onet_review))

print("ESCO keyword counts")
display(keyword_counts(esco_review))

# ----------------------------
# 6) Optional: inspect sample titles for each keyword
# ----------------------------
def show_samples(df, title_col, keyword, n=15):
    mask = df['matched_keywords'].apply(lambda xs: keyword in xs)
    return df.loc[mask, [title_col, 'matched_keywords_str']].drop_duplicates().head(n)

# Example:
display(show_samples(onet_review, onet_title_col, "software engineer"))
display(show_samples(esco_review, esco_title_col, "software engineer"))

# ----------------------------
# 7) Choose final keywords after inspection
#    Start with the important tech buckets.
# ----------------------------
final_keywords = [
    "software engineer",
    "software developer",
    "developer",
    "programmer",
    "computer",
    "information systems",
    "information technology",
    "ICT",
    "data scientist",
    "data engineer",
    "data analyst",
    "machine learning",
    "artificial intelligence",
    "cloud",
    "devops",
    "security",
    "cyber",
    "network",
    "database",
    "system administrator",
    "systems analyst",
    "quality assurance",
    "tester",
    "architect"
]

def keep_row(hit_list):
    return any(k in final_keywords for k in hit_list)

onet_clean = onet_review[onet_review['matched_keywords'].apply(keep_row)].copy()

# For ESCO, add a structural safety net:
# keep ICT-related groups as well, especially major group 25 and 35
esco_isco = esco_review['isco_group'].astype(str)
esco_structural_mask = esco_isco.str.startswith(('25', '35'))
esco_clean = esco_review[
    esco_review['matched_keywords'].apply(keep_row) | esco_structural_mask
].copy()

# ----------------------------
# 8) Save review and clean files
# ----------------------------
onet_review.to_csv('/content/onet_occupation_keyword_review.csv', index=False)
esco_review.to_csv('/content/esco_occupation_keyword_review.csv', index=False)

onet_clean.drop(columns=['title_norm', 'matched_keywords', 'matched_keywords_str']).to_csv(
    '/content/onet_occupations_tech.csv', index=False
)
esco_clean.drop(columns=['title_norm', 'matched_keywords', 'matched_keywords_str']).to_csv(
    '/content/esco_occupations_tech.csv', index=False
)

print("ONET clean rows:", len(onet_clean))
print("ESCO clean rows:", len(esco_clean))

ONET keyword counts


,keyword,count
1,computer,16
0,security,8
4,architect,6
3,network,3
10,tester,3
2,information systems,2
6,programmer,2
5,database,2
8,developer,2
7,software developer,1


ESCO keyword counts


,keyword,count
0,ICT,47
5,developer,23
4,tester,17
1,security,17
6,computer,16
9,architect,11
2,network,7
10,cloud,5
16,software developer,4
14,database,4


,title,matched_keywords_str


,esco_occ_label,matched_keywords_str


ONET clean rows: 36
ESCO clean rows: 171


Defining the roles to remove

In [ ]:
exclude_titles = [
    "Security Managers",
    "Security Management Specialists",
    "Geographic Information Systems Technologists and Technicians",
    "Architects, Except Landscape and Naval",
    "Landscape Architects",
    "Marine Engineers and Naval Architects",
    "Non-Destructive Testing Specialists",
    "First-Line Supervisors of Security Workers",
    "Security Guards",
    "Transportation Security Screeners",
    "Office Machine Operators, Except Computer",
    "Computer, Automated Teller, and Office Machine Repairers",
    "Security and Fire Alarm Systems Installers",
    "Inspectors, Testers, Sorters, Samplers, and Weighers",
    "Computer Numerically Controlled Tool Operators",
    "Computer Numerically Controlled Tool Programmers",
    "Computer User Support Specialists",
    "Electronics Engineers, Except Computer",
    "Computer Hardware Engineers",
    "Computer Science Teachers, Postsecondary"
]

Filtering the ONET occupation file

In [ ]:
import pandas as pd
import re

onet = pd.read_csv('/content/onet_occupations_tech.csv')

def norm(text):
    return re.sub(r'\s+', ' ', str(text).strip().lower())

exclude_set = set(norm(x) for x in exclude_titles)

onet['title_norm'] = onet['title'].apply(norm)

onet_clean = onet[~onet['title_norm'].isin(exclude_set)].copy()

onet_clean.to_csv('/content/onet_occupations_tech_clean.csv', index=False)

print("Original O*NET tech occupations:", len(onet))
print("Cleaned O*NET occupations:", len(onet_clean))

Original O*NET tech occupations: 36
Cleaned O*NET occupations: 16


Filtering the crosswalk using remaining ONET occupations

In [ ]:
crosswalk = pd.read_csv('/content/onet_esco_occ_crosswalk.csv')

crosswalk_clean = crosswalk[
    crosswalk['soc_code'].isin(onet_clean['soc_code'])
].copy()

crosswalk_clean.to_csv('/content/onet_esco_occ_crosswalk_tech_clean.csv', index=False)

print("Clean crosswalk rows:", len(crosswalk_clean))

Clean crosswalk rows: 78


Keeping only corresponding ESCO occupations

In [ ]:
esco = pd.read_csv('/content/esco_occupations.csv')

esco_clean = esco[
    esco['esco_occ_uri'].isin(crosswalk_clean['esco_occ_uri'])
].copy()

esco_clean.to_csv('/content/esco_occupations_tech_clean.csv', index=False)

print("Clean ESCO occupations:", len(esco_clean))

Clean ESCO occupations: 57


**Filtering ESCO skills using ESCO occs**

In [ ]:
# esco_rel = pd.read_csv('/content/esco_occ_requires_skill.csv')

# esco_rel_clean = esco_rel[
#     esco_rel['esco_occ_uri'].isin(esco_clean['esco_occ_uri'])
# ].copy()

# esco_rel_clean.to_csv('/content/esco_occ_requires_skill_tech_clean.csv', index=False)

Load files

In [ ]:
import pandas as pd
import re

esco_master = pd.read_csv('/content/esco_master.csv')
esco_occ_clean = pd.read_csv('/content/esco_occupations_tech_clean.csv')


Helper functions

In [ ]:
def norm_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

def first_non_empty(series):
    vals = [norm_text(v) for v in series if pd.notna(v) and norm_text(v) != ""]
    return vals[0] if vals else ""

def merge_unique_labels(series):
    vals = []
    for v in series:
        if pd.isna(v):
            continue
        parts = [p.strip() for p in re.split(r"[;|]", str(v)) if p.strip()]
        for p in parts:
            if p not in vals:
                vals.append(p)
    return "; ".join(vals)

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

Standardized key columns

In [ ]:
for col in ['occ_uri', 'skill_uri', 'skill_l0_uri', 'skill_l1_uri', 'skill_l2_uri', 'skill_l3_uri']:
    if col in esco_master.columns:
        esco_master[col] = esco_master[col].apply(norm_text)

clean_occ_uri_col = pick_col(esco_occ_clean, ['esco_occ_uri', 'occ_uri'])
if clean_occ_uri_col is None:
    raise KeyError("Could not find occupation URI column in esco_occupations_tech_clean.csv")

esco_occ_clean[clean_occ_uri_col] = esco_occ_clean[clean_occ_uri_col].apply(norm_text)

Filtering esco_master using cleaned ESCO occupations

In [ ]:
tech_occ_uris = set(esco_occ_clean[clean_occ_uri_col].dropna().astype(str))

tech_master = esco_master[esco_master['occ_uri'].isin(tech_occ_uris)].copy()
tech_master.to_csv('/content/final_esco_master_tech.csv', index=False)

print("Original esco_master rows:", len(esco_master))
print("Filtered tech master rows:", len(tech_master))
print("Unique tech occupations:", tech_master['occ_uri'].nunique())
print("Unique tech skills:", tech_master['skill_uri'].nunique())


Original esco_master rows: 127125
Filtered tech master rows: 4073
Unique tech occupations: 57
Unique tech skills: 839


Building final ESCO skills file

In [ ]:
skill_cols = [
    'skill_uri',
    'skill_preferred_label',
    'skill_alternative_label',
    'skill_type',
    'is_digital',
    'is_transversal',
    'is_digcomp'
]
skill_cols_existing = [c for c in skill_cols if c in tech_master.columns]

skills_group = tech_master[skill_cols_existing].copy()
skills_group = skills_group.dropna(subset=['skill_uri'])

agg_map = {}
for col in skill_cols_existing:
    if col == 'skill_uri':
        continue
    elif col == 'skill_alternative_label':
        agg_map[col] = merge_unique_labels
    else:
        agg_map[col] = first_non_empty

final_esco_skills_tech = (
    skills_group
    .groupby('skill_uri', as_index=False)
    .agg(agg_map)
    .drop_duplicates(subset=['skill_uri'])
    .copy()
)

final_esco_skills_tech = final_esco_skills_tech.rename(columns={
    'skill_uri': 'esco_skill_uri',
    'skill_preferred_label': 'label'
})

skill_order = ['esco_skill_uri', 'label', 'skill_alternative_label', 'skill_type', 'is_digital', 'is_transversal', 'is_digcomp']
skill_order = [c for c in skill_order if c in final_esco_skills_tech.columns]
final_esco_skills_tech = final_esco_skills_tech[skill_order]

final_esco_skills_tech.to_csv('/content/final_esco_skills_tech.csv', index=False)
print("Saved final_esco_skills_tech.csv:", len(final_esco_skills_tech))


Saved final_esco_skills_tech.csv: 839


Build occupation-skill relationship file

In [ ]:
rel_cols = ['occ_uri', 'skill_uri', 'relation_type', 'skill_type_in_occ']
rel_cols_existing = [c for c in rel_cols if c in tech_master.columns]

final_esco_occ_requires_skill_tech = (
    tech_master[rel_cols_existing]
    .dropna(subset=['occ_uri', 'skill_uri'])
    .drop_duplicates(subset=['occ_uri', 'skill_uri', 'relation_type', 'skill_type_in_occ'])
    .copy()
)

final_esco_occ_requires_skill_tech = final_esco_occ_requires_skill_tech.rename(columns={
    'occ_uri': 'esco_occ_uri',
    'skill_uri': 'esco_skill_uri'
})

rel_order = ['esco_occ_uri', 'esco_skill_uri', 'relation_type', 'skill_type_in_occ']
rel_order = [c for c in rel_order if c in final_esco_occ_requires_skill_tech.columns]
final_esco_occ_requires_skill_tech = final_esco_occ_requires_skill_tech[rel_order]

final_esco_occ_requires_skill_tech.to_csv('/content/final_esco_occ_requires_skill_tech.csv', index=False)
print("Saved final_esco_occ_requires_skill_tech.csv:", len(final_esco_occ_requires_skill_tech))


Saved final_esco_occ_requires_skill_tech.csv: 4061


Build skill categories from hierarchy

In [ ]:
cat_nodes = []

# Level 0 categories
if 'skill_l0_uri' in tech_master.columns and 'skill_l0_label' in tech_master.columns:
    l0_cols = ['skill_l0_uri', 'skill_l0_label']
    if 'skill_l0_code' in tech_master.columns:
        l0_cols.append('skill_l0_code')

    l0 = tech_master[l0_cols].copy()
    l0 = l0.dropna(subset=['skill_l0_uri', 'skill_l0_label']).drop_duplicates(subset=['skill_l0_uri']).copy()
    l0 = l0.rename(columns={
        'skill_l0_uri': 'category_uri',
        'skill_l0_label': 'name',
        'skill_l0_code': 'code'
    })
    l0['level'] = 0
    l0['parent_uri'] = ""
    l0['parent_name'] = ""
    cat_nodes.append(l0)

# Level 1 categories
if 'skill_l1_uri' in tech_master.columns and 'skill_l1_label' in tech_master.columns:
    l1_cols = ['skill_l1_uri', 'skill_l1_label']
    if 'skill_l1_code' in tech_master.columns:
        l1_cols.append('skill_l1_code')
    if 'skill_l0_uri' in tech_master.columns:
        l1_cols.append('skill_l0_uri')
    if 'skill_l0_label' in tech_master.columns:
        l1_cols.append('skill_l0_label')

    l1 = tech_master[l1_cols].copy()
    l1 = l1.dropna(subset=['skill_l1_uri', 'skill_l1_label']).drop_duplicates(subset=['skill_l1_uri']).copy()
    l1 = l1.rename(columns={
        'skill_l1_uri': 'category_uri',
        'skill_l1_label': 'name',
        'skill_l1_code': 'code',
        'skill_l0_uri': 'parent_uri',
        'skill_l0_label': 'parent_name'
    })
    l1['level'] = 1
    if 'parent_uri' not in l1.columns:
        l1['parent_uri'] = ""
    if 'parent_name' not in l1.columns:
        l1['parent_name'] = ""
    cat_nodes.append(l1)

final_skill_categories_tech = pd.concat(cat_nodes, ignore_index=True).drop_duplicates(subset=['category_uri']).copy()

for col in ['code', 'parent_uri', 'parent_name']:
    if col not in final_skill_categories_tech.columns:
        final_skill_categories_tech[col] = ""

final_skill_categories_tech = final_skill_categories_tech[
    ['category_uri', 'name', 'code', 'level', 'parent_uri', 'parent_name']
]

final_skill_categories_tech.to_csv('/content/final_skill_categories_tech.csv', index=False)
print("Saved final_skill_categories_tech.csv:", len(final_skill_categories_tech))

# Category hierarchy edges
final_skill_category_edges_tech = final_skill_categories_tech[
    (final_skill_categories_tech['level'] == 1) &
    (final_skill_categories_tech['parent_uri'].astype(str).str.len() > 0)
][['category_uri', 'parent_uri']].drop_duplicates().copy()

final_skill_category_edges_tech = final_skill_category_edges_tech.rename(columns={
    'category_uri': 'child_category_uri',
    'parent_uri': 'parent_category_uri'
})

final_skill_category_edges_tech.to_csv('/content/final_skill_category_edges_tech.csv', index=False)
print("Saved final_skill_category_edges_tech.csv:", len(final_skill_category_edges_tech))


Saved final_skill_categories_tech.csv: 22
Saved final_skill_category_edges_tech.csv: 19


**preview**

In [ ]:
print("\nPreview: final_esco_skills_tech.csv")
display(final_esco_skills_tech.head(10))

print("\nPreview: final_esco_occ_requires_skill_tech.csv")
display(final_esco_occ_requires_skill_tech.head(10))

print("\nPreview: final_skill_categories_tech.csv")
display(final_skill_categories_tech.head(10))


Preview: final_esco_skills_tech.csv


,esco_skill_uri,label,skill_alternative_label,skill_type,is_digital,is_transversal,is_digcomp
0,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell techniques,knowledge,True,False,False
1,http://data.europa.eu/esco/skill/00b9a3aa-7070...,draft scientific or academic papers and techni...,create technical documentation\ncreate technic...,skill/competence,False,False,False
2,http://data.europa.eu/esco/skill/00c04e40-35ea...,incremental development,gradual development,knowledge,True,False,False
3,http://data.europa.eu/esco/skill/013441c1-1f13...,KDevelop,kdevplatform\nKDevelop platform,knowledge,True,False,False
4,http://data.europa.eu/esco/skill/01d269d9-b058...,develop blockchain innovative architectures,build innovative blockchain architectures,skill/competence,True,False,False
5,http://data.europa.eu/esco/skill/02058de6-4b98...,network engineering,network architecture,knowledge,True,False,False
6,http://data.europa.eu/esco/skill/020918ef-4b55...,migrate existing data,drift existing data,skill/competence,True,False,False
7,http://data.europa.eu/esco/skill/020b3c27-bae1...,manage supplies,supplies managing\ncontrol and monitor supplie...,skill/competence,False,False,False
8,http://data.europa.eu/esco/skill/022dc430-872a...,define database physical structure,define database physical storage structure,skill/competence,True,False,False
9,http://data.europa.eu/esco/skill/034c29fa-c3ba...,Erlang,Erlang programming language,knowledge,True,False,False



Preview: final_esco_occ_requires_skill_tech.csv


,esco_occ_uri,esco_skill_uri,relation_type,skill_type_in_occ
1649,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/09f2f811-a3fb...,essential,knowledge
1650,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/0da6cab1-0ee9...,essential,knowledge
1651,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/1ca140f8-d8aa...,essential,knowledge
1652,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/2ddb1226-1117...,essential,knowledge
1653,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/32428b21-514b...,essential,knowledge
1654,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/557d1221-0235...,essential,knowledge
1655,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/60b44dab-03f1...,essential,knowledge
1656,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/6e4f75b4-c60f...,essential,knowledge
1657,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/7f2afce6-e98b...,essential,knowledge
1658,http://data.europa.eu/esco/occupation/02eb0ae6...,http://data.europa.eu/esco/skill/c7cc6dc5-d56b...,essential,knowledge



Preview: final_skill_categories_tech.csv


,category_uri,name,code,level,parent_uri,parent_name
0,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,K,0,,
1,http://data.europa.eu/esco/skill/335228d2-297d...,skills,S,0,,
2,http://data.europa.eu/esco/skill/04a13491-b58c...,transversal skills and competences,T,0,,
3,http://data.europa.eu/esco/isced-f/06,information and communication technologies (icts),06,1,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge
4,http://data.europa.eu/esco/isced-f/07,"engineering, manufacturing and construction",07,1,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge
5,http://data.europa.eu/esco/isced-f/04,"business, administration and law",04,1,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge
6,http://data.europa.eu/esco/skill/9b8bb484-dcba...,working with machinery and specialised equipment,S8,1,http://data.europa.eu/esco/skill/335228d2-297d...,skills
7,http://data.europa.eu/esco/skill/dc06de9f-dd3a...,"communication, collaboration and creativity",S1,1,http://data.europa.eu/esco/skill/335228d2-297d...,skills
8,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills,S2,1,http://data.europa.eu/esco/skill/335228d2-297d...,skills
9,http://data.europa.eu/esco/skill/243eb885-07c7...,working with computers,S5,1,http://data.europa.eu/esco/skill/335228d2-297d...,skills


# **Scraping COURSERA courses for esco skills**

In [ ]:
!pip -q install pandas requests beautifulsoup4 tqdm

In [ ]:
import pandas as pd
import json
import re
import time
import random
from urllib.parse import quote_plus, urljoin
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm

skills = pd.read_csv("/content/final_esco_skills_tech.csv")

# Support both possible column names
skill_col = "label" if "label" in skills.columns else "skill_preferred_label"
skills = skills[[c for c in ["esco_skill_uri", skill_col, "skill_alternative_label"] if c in skills.columns]].copy()
skills = skills.rename(columns={skill_col: "skill_label"})

skills.head()

,esco_skill_uri,skill_label,skill_alternative_label
0,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell techniques
1,http://data.europa.eu/esco/skill/00b9a3aa-7070...,draft scientific or academic papers and techni...,create technical documentation\ncreate technic...
2,http://data.europa.eu/esco/skill/00c04e40-35ea...,incremental development,gradual development
3,http://data.europa.eu/esco/skill/013441c1-1f13...,KDevelop,kdevplatform\nKDevelop platform
4,http://data.europa.eu/esco/skill/01d269d9-b058...,develop blockchain innovative architectures,build innovative blockchain architectures


In [ ]:
BASE_URL = "https://www.coursera.org/courses"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36"
}

def clean_text(x):
    return re.sub(r"\s+", " ", str(x)).strip()

def infer_program_type(href: str) -> str:
    href = href.lower()
    if "/learn/" in href:
        return "course"
    if "/specializations/" in href:
        return "specialization"
    if "/professional-certificates/" in href:
        return "professional_certificate"
    if "/certificates/" in href:
        return "certificate"
    return "other"

def infer_level(text: str) -> str:
    t = text.lower()
    if "beginner" in t:
        return "beginner"
    if "intermediate" in t:
        return "intermediate"
    if "advanced" in t:
        return "advanced"
    if "mixed" in t:
        return "mixed"
    return ""

def extract_course_cards(html: str, max_results: int = 5):
    soup = BeautifulSoup(html, "html.parser")
    results = []
    seen = set()

    # Broad, resilient strategy: find relevant learning links anywhere in the page.
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not re.match(r"^/(learn|specializations|professional-certificates|certificates)/", href):
            continue

        url = urljoin("https://www.coursera.org", href.split("?")[0])
        if url in seen:
            continue

        title = clean_text(a.get_text(" ", strip=True))
        if not title or len(title) < 4:
            continue

        # Try to capture nearby card text for metadata.
        parent = a
        for _ in range(3):
            parent = parent.parent if parent else None
        card_text = clean_text(parent.get_text(" ", strip=True)) if parent else title

        results.append({
            "title": title,
            "url": url,
            "program_type": infer_program_type(href),
            "level": infer_level(card_text),
            "card_text": card_text
        })
        seen.add(url)

        if len(results) >= max_results:
            break

    return results

def scrape_coursera_for_skill(skill_label: str, max_results: int = 5, timeout: int = 30):
    query = quote_plus(skill_label)
    url = f"{BASE_URL}?query={query}"
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()

    raw = extract_course_cards(resp.text, max_results=max_results)

    # Light post-processing
    out = []
    for item in raw:
        txt = item.pop("card_text", "")
        provider = ""
        # Heuristic: many result cards include provider name near the title in the card text.
        # Keep this blank if we cannot detect it reliably.
        m = re.search(r"\bBy\s+([A-Z][A-Za-z0-9&.,'() \-]{2,80})\b", txt)
        if m:
            provider = clean_text(m.group(1))

        item["provider"] = provider
        out.append(item)

    return out

In [ ]:
output_path = "/content/coursera_skill_catalog.json"
checkpoint_path = "/content/coursera_skill_catalog_checkpoint.jsonl"
processed_path = "/content/coursera_processed_skills.txt"

# Resume support
processed = set()
try:
    with open(processed_path, "r", encoding="utf-8") as f:
        processed = set(line.strip() for line in f if line.strip())
except FileNotFoundError:
    pass

skill_cache = {}
global_seen_urls = set()

# Load existing checkpoint if present
try:
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            skill_cache[row["esco_skill_uri"]] = row
            for r in row["results"]:
                global_seen_urls.add(r["url"])
except FileNotFoundError:
    pass

for _, row in tqdm(skills.iterrows(), total=len(skills)):
    esco_skill_uri = row["esco_skill_uri"]
    skill_label = row["skill_label"]

    if esco_skill_uri in processed:
        continue

    try:
        results = scrape_coursera_for_skill(skill_label, max_results=5)

        # Global dedupe across all skills
        deduped = []
        for r in results:
            if r["url"] in global_seen_urls:
                continue
            global_seen_urls.add(r["url"])
            deduped.append(r)

        record = {
            "esco_skill_uri": esco_skill_uri,
            "skill_label": skill_label,
            "query": skill_label,
            "results": deduped,
            "result_count": len(deduped)
        }

        skill_cache[esco_skill_uri] = record

        # Incremental checkpoint
        with open(checkpoint_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        with open(processed_path, "a", encoding="utf-8") as f:
            f.write(esco_skill_uri + "\n")

    except Exception as e:
        record = {
            "esco_skill_uri": esco_skill_uri,
            "skill_label": skill_label,
            "query": skill_label,
            "results": [],
            "result_count": 0,
            "error": str(e)
        }
        skill_cache[esco_skill_uri] = record

        with open(checkpoint_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        with open(processed_path, "a", encoding="utf-8") as f:
            f.write(esco_skill_uri + "\n")

    time.sleep(random.uniform(1.0, 2.2))

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(list(skill_cache.values()), f, ensure_ascii=False, indent=2)

print(f"Saved: {output_path}")
print(f"Saved incremental checkpoint: {checkpoint_path}")

100%|██████████| 839/839 [38:03<00:00,  2.72s/it]

Saved: /content/coursera_skill_catalog.json
Saved incremental checkpoint: /content/coursera_skill_catalog_checkpoint.jsonl


In [ ]:
with open("/content/coursera_skill_catalog.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Skills with at least one result
hits = [x for x in data if x.get("result_count", 0) > 0]
misses = [x for x in data if x.get("result_count", 0) == 0]

print("Skills with matches:", len(hits))
print("Skills with no matches:", len(misses))

# Example preview
hits[:2]

Skills with matches: 634
Skills with no matches: 205


[{'esco_skill_uri': 'http://data.europa.eu/esco/skill/000f1d3d-220f-4789-9c0a-cc742521fb02',
  'skill_label': 'Haskell',
  'query': 'Haskell',
  'results': [{'title': 'Rust Programming',
    'url': 'https://www.coursera.org/specializations/rust-programming',
    'program_type': 'specialization',
    'level': 'beginner',
    'provider': ''},
   {'title': 'Functional Programming in Scala',
    'url': 'https://www.coursera.org/specializations/scala',
    'program_type': 'specialization',
    'level': 'intermediate',
    'provider': ''},
   {'title': 'AI Agents with Model Context Protocol & Typescript',
    'url': 'https://www.coursera.org/specializations/ai-agents-mode-context-protocol',
    'program_type': 'specialization',
    'level': 'beginner',
    'provider': ''},
   {'title': 'C, Go, and C++: A Comprehensive Introduction to Programming',
    'url': 'https://www.coursera.org/specializations/c-go-c-plus-plus',
    'program_type': 'specialization',
    'level': 'intermediate',
    'pr

# **Scraping Google Certification pages to get the certification data**

Imports

In [ ]:
!pip -q install pandas requests beautifulsoup4 tqdm

import pandas as pd
import requests
import json
import re
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin

Loading CustomRole list

In [ ]:
custom_roles = [
    "Software Engineer",
    "Backend Engineer",
    "Frontend Engineer",
    "Full-Stack Engineer",
    "Platform Engineer",
    "DevOps Engineer",
    "Site Reliability Engineer",
    "Cloud Engineer",
    "Security Engineer",
    "QA Automation Engineer",
    "Machine Learning Engineer",
    "AI Engineer",
    "MLOps Engineer",
    "Data Engineer",
    "Data Scientist",
]

roles_df = pd.DataFrame({"role_title": custom_roles})

Google certfication URLs

In [ ]:
CERT_URLS = {
    "cloud_digital_leader": {
        "cert_name": "Cloud Digital Leader",
        "tier": "foundational",
        "url": "https://cloud.google.com/learn/certification/cloud-digital-leader/"
    },
    "generative_ai_leader": {
        "cert_name": "Generative AI Leader",
        "tier": "foundational",
        "url": "https://cloud.google.com/learn/certification/generative-ai-leader"
    },
    "associate_cloud_engineer": {
        "cert_name": "Associate Cloud Engineer",
        "tier": "associate",
        "url": "https://cloud.google.com/learn/certification/cloud-engineer/"
    },
    "associate_data_practitioner": {
        "cert_name": "Associate Data Practitioner",
        "tier": "associate",
        "url": "https://cloud.google.com/learn/certification/data-practitioner"
    },
    "associate_workspace_admin": {
        "cert_name": "Associate Google Workspace Administrator",
        "tier": "associate",
        "url": "https://cloud.google.com/learn/certification/associate-google-workspace-administrator"
    },
    "professional_cloud_architect": {
        "cert_name": "Professional Cloud Architect",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/cloud-architect"
    },
    "professional_cloud_developer": {
        "cert_name": "Professional Cloud Developer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/cloud-developer"
    },
    "professional_data_engineer": {
        "cert_name": "Professional Data Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/data-engineer"
    },
    "professional_cloud_database_engineer": {
        "cert_name": "Professional Cloud Database Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/cloud-database-engineer"
    },
    "professional_cloud_devops_engineer": {
        "cert_name": "Professional Cloud DevOps Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/cloud-devops-engineer"
    },
    "professional_cloud_security_engineer": {
        "cert_name": "Professional Cloud Security Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/cloud-security-engineer"
    },
    "professional_cloud_network_engineer": {
        "cert_name": "Professional Cloud Network Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/cloud-network-engineer"
    },
    "professional_ml_engineer": {
        "cert_name": "Professional Machine Learning Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/machine-learning-engineer"
    },
    "professional_security_ops_engineer": {
        "cert_name": "Professional Security Operations Engineer",
        "tier": "professional",
        "url": "https://cloud.google.com/learn/certification/security-operations-engineer"
    },
}

**Role -> certification mapping**  
Keeping it small: 2-3 certs per role

In [ ]:
ROLE_TO_CERTS = {
    "Software Engineer": [
        "associate_cloud_engineer",
        "professional_cloud_developer",
        "professional_cloud_architect",
    ],
    "Backend Engineer": [
        "professional_cloud_developer",
        "professional_cloud_architect",
        "associate_cloud_engineer",
    ],
    "Frontend Engineer": [
        "professional_cloud_developer",
        "associate_cloud_engineer",
        "cloud_digital_leader",
    ],
    "Full-Stack Engineer": [
        "professional_cloud_developer",
        "professional_cloud_architect",
        "associate_cloud_engineer",
    ],
    "Platform Engineer": [
        "professional_cloud_architect",
        "professional_cloud_devops_engineer",
        "associate_cloud_engineer",
    ],
    "DevOps Engineer": [
        "professional_cloud_devops_engineer",
        "associate_cloud_engineer",
        "professional_cloud_architect",
    ],
    "Site Reliability Engineer": [
        "professional_cloud_devops_engineer",
        "professional_cloud_network_engineer",
        "associate_cloud_engineer",
    ],
    "Cloud Engineer": [
        "associate_cloud_engineer",
        "professional_cloud_architect",
        "professional_cloud_network_engineer",
    ],
    "Security Engineer": [
        "professional_cloud_security_engineer",
        "professional_security_ops_engineer",
        "professional_cloud_network_engineer",
    ],
    "QA Automation Engineer": [
        "professional_cloud_developer",
        "associate_cloud_engineer",
        "cloud_digital_leader",
    ],
    "Machine Learning Engineer": [
        "professional_ml_engineer",
        "generative_ai_leader",
        "professional_data_engineer",
    ],
    "AI Engineer": [
        "generative_ai_leader",
        "professional_ml_engineer",
        "cloud_digital_leader",
    ],
    "MLOps Engineer": [
        "professional_ml_engineer",
        "professional_cloud_devops_engineer",
        "professional_data_engineer",
    ],
    "Data Engineer": [
        "professional_data_engineer",
        "professional_cloud_database_engineer",
        "associate_data_practitioner",
    ],
    "Data Scientist": [
        "professional_data_engineer",
        "associate_data_practitioner",
        "professional_ml_engineer",
    ],
}

Scraper helpers

In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0 Safari/537.36"
    )
}

def clean_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def get_meta_description(soup):
    meta = soup.find("meta", attrs={"name": "description"})
    return meta.get("content", "").strip() if meta else ""

def extract_section_text(page_text, heading):
    """
    Light heuristic to capture text after a heading.
    Works well enough for a cached metadata pass.
    """
    pattern = rf"(?is){re.escape(heading)}\s*(.*?)(?:\n[A-Z][^\n]{{2,80}}\n|$)"
    m = re.search(pattern, page_text)
    return clean_text(m.group(1)) if m else ""

def scrape_cert_page(url):
    resp = requests.get(url, headers=HEADERS, timeout=40)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    page_text = clean_text(soup.get_text("\n", strip=True))

    h1 = soup.find("h1")
    title = clean_text(h1.get_text(" ", strip=True)) if h1 else ""

    data = {
        "url": url,
        "page_title": title,
        "meta_description": get_meta_description(soup),
        "recommended_candidate": "",
        "prerequisites": "",
        "preparation": "",
        "exam_focus": "",
        "validity": "",
        "exam_delivery": "",
        "source": "cloud.google.com",
    }

    # Heuristic section extraction from text. Good enough for a JSON cache.
    for heading, field in [
        ("Recommended candidate", "recommended_candidate"),
        ("Prerequisites", "prerequisites"),
        ("Preparation", "preparation"),
        ("Preparation:", "preparation"),
        ("Exam delivery method", "exam_delivery"),
        ("Validity period", "validity"),
        ("The exam assesses your ability to", "exam_focus"),
        ("The exam measures your ability to", "exam_focus"),
    ]:
        if not data[field]:
            data[field] = extract_section_text(page_text, heading)

    return data

Building 1 JSON record per role with certification data

In [ ]:
results = []

for _, row in tqdm(roles_df.iterrows(), total=len(roles_df)):
    role = row["role_title"]
    cert_keys = ROLE_TO_CERTS.get(role, [])

    role_record = {
        "custom_role": role,
        "certifications": []
    }

    for cert_key in cert_keys:
        cert_info = CERT_URLS[cert_key]
        try:
            cert_page = scrape_cert_page(cert_info["url"])
            cert_record = {
                "cert_key": cert_key,
                "cert_name": cert_info["cert_name"],
                "tier": cert_info["tier"],
                "url": cert_info["url"],
                "page_title": cert_page["page_title"],
                "meta_description": cert_page["meta_description"],
                "recommended_candidate": cert_page["recommended_candidate"],
                "prerequisites": cert_page["prerequisites"],
                "preparation": cert_page["preparation"],
                "exam_focus": cert_page["exam_focus"],
                "validity": cert_page["validity"],
                "exam_delivery": cert_page["exam_delivery"],
            }
        except Exception as e:
            cert_record = {
                "cert_key": cert_key,
                "cert_name": cert_info["cert_name"],
                "tier": cert_info["tier"],
                "url": cert_info["url"],
                "error": str(e)
            }

        role_record["certifications"].append(cert_record)

        # Pause for politeness
        time.sleep(random.uniform(0.8, 1.8))

    results.append(role_record)

100%|██████████| 15/15 [01:35<00:00,  6.35s/it]


**Saving to JSON file**

In [ ]:
output_path = "/content/google_cloud_certifications_by_role.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved to {output_path}")

Saved to /content/google_cloud_certifications_by_role.json


# Creating **lvl2** and **lvl3** skill category file

Load files

In [ ]:
import pandas as pd
import re

master = pd.read_csv('/content/final_esco_master_tech.csv')
skills = pd.read_csv('/content/final_esco_skills_tech.csv')

# Support either column naming style
skill_uri_col = 'esco_skill_uri' if 'esco_skill_uri' in skills.columns else 'skill_uri'
skill_uris = set(skills[skill_uri_col].astype(str).str.strip())

Helper functions

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

def pick_first_non_empty(row, cols):
    for c in cols:
        if c in row and pd.notna(row[c]) and clean_text(row[c]) != "":
            return clean_text(row[c])
    return ""

# Clean relevant master columns
for col in [
    'skill_uri',
    'skill_l1_uri', 'skill_l1_label', 'skill_l1_code',
    'skill_l2_uri', 'skill_l2_label', 'skill_l2_code',
    'skill_l3_uri', 'skill_l3_label', 'skill_l3_code'
]:
    if col in master.columns:
        master[col] = master[col].apply(clean_text)

# Filter to only tech skills already in final_esco_skills_tech.csv
m = master[master['skill_uri'].isin(skill_uris)].copy()

print("Filtered master rows:", len(m))
print("Unique skills:", m['skill_uri'].nunique())

Filtered master rows: 4073
Unique skills: 839


1. Building specific category nodes from level 2 and level 3  
2. Keep parent_uri so hierarchy can be linked to existing l1 nodes.

In [ ]:
cat_parts = []

# Level 2 categories
if 'skill_l2_uri' in m.columns and 'skill_l2_label' in m.columns:
    cols = ['skill_l2_uri', 'skill_l2_label']
    if 'skill_l2_code' in m.columns:
        cols.append('skill_l2_code')
    if 'skill_l1_uri' in m.columns:
        cols.append('skill_l1_uri')
    if 'skill_l1_label' in m.columns:
        cols.append('skill_l1_label')

    lvl2 = m[cols].copy()
    lvl2 = lvl2.dropna(subset=['skill_l2_uri', 'skill_l2_label'])
    lvl2 = lvl2.drop_duplicates(subset=['skill_l2_uri']).copy()
    lvl2 = lvl2.rename(columns={
        'skill_l2_uri': 'category_uri',
        'skill_l2_label': 'name',
        'skill_l2_code': 'code',
        'skill_l1_uri': 'parent_uri',
        'skill_l1_label': 'parent_name'
    })
    lvl2['level'] = 2
    if 'code' not in lvl2.columns:
        lvl2['code'] = ""
    if 'parent_uri' not in lvl2.columns:
        lvl2['parent_uri'] = ""
    if 'parent_name' not in lvl2.columns:
        lvl2['parent_name'] = ""
    cat_parts.append(lvl2)

# Level 3 categories
if 'skill_l3_uri' in m.columns and 'skill_l3_label' in m.columns:
    cols = ['skill_l3_uri', 'skill_l3_label']
    if 'skill_l3_code' in m.columns:
        cols.append('skill_l3_code')
    if 'skill_l2_uri' in m.columns:
        cols.append('skill_l2_uri')
    if 'skill_l2_label' in m.columns:
        cols.append('skill_l2_label')

    lvl3 = m[cols].copy()
    lvl3 = lvl3.dropna(subset=['skill_l3_uri', 'skill_l3_label'])
    lvl3 = lvl3.drop_duplicates(subset=['skill_l3_uri']).copy()
    lvl3 = lvl3.rename(columns={
        'skill_l3_uri': 'category_uri',
        'skill_l3_label': 'name',
        'skill_l3_code': 'code',
        'skill_l2_uri': 'parent_uri',
        'skill_l2_label': 'parent_name'
    })
    lvl3['level'] = 3
    if 'code' not in lvl3.columns:
        lvl3['code'] = ""
    if 'parent_uri' not in lvl3.columns:
        lvl3['parent_uri'] = ""
    if 'parent_name' not in lvl3.columns:
        lvl3['parent_name'] = ""
    cat_parts.append(lvl3)

specific_categories = pd.concat(cat_parts, ignore_index=True).drop_duplicates(subset=['category_uri']).copy()

# Normalize column order
for col in ['code', 'parent_uri', 'parent_name']:
    if col not in specific_categories.columns:
        specific_categories[col] = ""

specific_categories = specific_categories[
    ['category_uri', 'name', 'code', 'level', 'parent_uri', 'parent_name']
]

# Save specific category nodes
specific_categories_path = '/content/final_skill_categories_specific_l2_l3.csv'
specific_categories.to_csv(specific_categories_path, index=False)

print("Saved:", specific_categories_path)
print("Specific categories:", len(specific_categories))

Saved: /content/final_skill_categories_specific_l2_l3.csv
Specific categories: 211


Building category hierarchy edges  
Level 3 -> Level 2  
Level 2 -> Level 1 (parent nodes already exist in Neo4j)

In [ ]:
category_edges = []

# Level 3 -> Level 2
if 'skill_l3_uri' in m.columns and 'skill_l2_uri' in m.columns:
    e3 = m[['skill_l3_uri', 'skill_l2_uri']].copy()
    e3 = e3.dropna(subset=['skill_l3_uri', 'skill_l2_uri'])
    e3 = e3.drop_duplicates()
    e3 = e3.rename(columns={
        'skill_l3_uri': 'child_category_uri',
        'skill_l2_uri': 'parent_category_uri'
    })
    category_edges.append(e3)

# Level 2 -> Level 1
if 'skill_l2_uri' in m.columns and 'skill_l1_uri' in m.columns:
    e2 = m[['skill_l2_uri', 'skill_l1_uri']].copy()
    e2 = e2.dropna(subset=['skill_l2_uri', 'skill_l1_uri'])
    e2 = e2.drop_duplicates()
    e2 = e2.rename(columns={
        'skill_l2_uri': 'child_category_uri',
        'skill_l1_uri': 'parent_category_uri'
    })
    category_edges.append(e2)

specific_edges = pd.concat(category_edges, ignore_index=True).drop_duplicates().copy()

specific_edges_path = '/content/final_skill_category_edges_specific_l2_l3.csv'
specific_edges.to_csv(specific_edges_path, index=False)

print("Saved:", specific_edges_path)
print("Category edges:", len(specific_edges))

Saved: /content/final_skill_category_edges_specific_l2_l3.csv
Category edges: 216


Build skill -> specific category mapping  
Prefer l3, else l2, else l1 as fallback

In [ ]:
skill_category_map = []

for _, row in m.iterrows():
    chosen_uri = ""
    chosen_label = ""
    chosen_level = None

    # Prefer deepest available
    if 'skill_l3_uri' in m.columns and row.get('skill_l3_uri', ""):
        chosen_uri = clean_text(row['skill_l3_uri'])
        chosen_label = clean_text(row.get('skill_l3_label', ""))
        chosen_level = 3
    elif 'skill_l2_uri' in m.columns and row.get('skill_l2_uri', ""):
        chosen_uri = clean_text(row['skill_l2_uri'])
        chosen_label = clean_text(row.get('skill_l2_label', ""))
        chosen_level = 2
    elif 'skill_l1_uri' in m.columns and row.get('skill_l1_uri', ""):
        chosen_uri = clean_text(row['skill_l1_uri'])
        chosen_label = clean_text(row.get('skill_l1_label', ""))
        chosen_level = 1

    if chosen_uri:
        skill_category_map.append({
            "esco_skill_uri": row["skill_uri"],
            "skill_label": row.get("skill_preferred_label", row.get("label", "")),
            "category_uri": chosen_uri,
            "category_label": chosen_label,
            "category_level": chosen_level
        })

skill_category_map_df = pd.DataFrame(skill_category_map).drop_duplicates(
    subset=['esco_skill_uri', 'category_uri']
).copy()

skill_category_map_path = '/content/final_esco_skill_to_specific_category_l2_l3.csv'
skill_category_map_df.to_csv(skill_category_map_path, index=False)

print("Saved:", skill_category_map_path)
print("Skill-category mappings:", len(skill_category_map_df))

Saved: /content/final_esco_skill_to_specific_category_l2_l3.csv
Skill-category mappings: 839


Quick preview

In [ ]:
print("\nPreview specific categories:")
display(specific_categories.head(10))

print("\nPreview mapping:")
display(skill_category_map_df.head(10))


Preview specific categories:


,category_uri,name,code,level,parent_uri,parent_name
0,http://data.europa.eu/esco/isced-f/061,information and communication technologies (icts),061,2,http://data.europa.eu/esco/isced-f/06,information and communication technologies (icts)
1,http://data.europa.eu/esco/isced-f/071,engineering and engineering trades,071,2,http://data.europa.eu/esco/isced-f/07,"engineering, manufacturing and construction"
2,http://data.europa.eu/esco/isced-f/041,business and administration,041,2,http://data.europa.eu/esco/isced-f/04,"business, administration and law"
3,http://data.europa.eu/esco/skill/4d7fb3eb-554a...,using precision instrumentation and equipment,S8.6,2,http://data.europa.eu/esco/skill/9b8bb484-dcba...,working with machinery and specialised equipment
4,http://data.europa.eu/esco/skill/0a295163-3e17...,designing systems and products,S1.11,2,http://data.europa.eu/esco/skill/dc06de9f-dd3a...,"communication, collaboration and creativity"
5,http://data.europa.eu/esco/skill/5770f630-9b25...,teaching and training,S1.3,2,http://data.europa.eu/esco/skill/dc06de9f-dd3a...,"communication, collaboration and creativity"
6,http://data.europa.eu/esco/skill/587e1cd6-a376...,processing information,S2.4,2,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills
7,http://data.europa.eu/esco/skill/e173eaf8-c99b...,calculating and estimating,S2.6,2,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills
8,http://data.europa.eu/esco/skill/6889a21c-837b...,obtaining information verbally,S1.7,2,http://data.europa.eu/esco/skill/dc06de9f-dd3a...,"communication, collaboration and creativity"
9,http://data.europa.eu/esco/skill/75c1ab73-5b00...,setting up and protecting computer systems,S5.2,2,http://data.europa.eu/esco/skill/243eb885-07c7...,working with computers



Preview mapping:


,esco_skill_uri,skill_label,category_uri,category_label,category_level
0,http://data.europa.eu/esco/skill/09f2f811-a3fb...,systems development life-cycle,http://data.europa.eu/esco/isced-f/0613,software and applications development and anal...,3
1,http://data.europa.eu/esco/skill/0da6cab1-0ee9...,ICT network routing,http://data.europa.eu/esco/isced-f/0612,database and network design and administration,3
2,http://data.europa.eu/esco/skill/1ca140f8-d8aa...,microwave principles,http://data.europa.eu/esco/isced-f/0714,electronics and automation,3
3,http://data.europa.eu/esco/skill/2ddb1226-1117...,ICT network security risks,http://data.europa.eu/esco/isced-f/0612,database and network design and administration,3
4,http://data.europa.eu/esco/skill/32428b21-514b...,procurement of ICT network equipment,http://data.europa.eu/esco/isced-f/0612,database and network design and administration,3
5,http://data.europa.eu/esco/skill/557d1221-0235...,electronics principles,http://data.europa.eu/esco/isced-f/0714,electronics and automation,3
6,http://data.europa.eu/esco/skill/60b44dab-03f1...,telecommunication industry,http://data.europa.eu/esco/isced-f/0714,electronics and automation,3
7,http://data.europa.eu/esco/skill/6e4f75b4-c60f...,ICT communications protocols,http://data.europa.eu/esco/isced-f/0714,electronics and automation,3
8,http://data.europa.eu/esco/skill/7f2afce6-e98b...,telecom regulations,http://data.europa.eu/esco/isced-f/0714,electronics and automation,3
9,http://data.europa.eu/esco/skill/c7cc6dc5-d56b...,quality assurance methodologies,http://data.europa.eu/esco/isced-f/0417,work skills,3


# **Flatten json files to csv**

In [ ]:
!pip -q install pandas

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

Helper functions

In [ ]:
def clean_text(x):
    if x is None:
        return ""
    text = str(x)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_section(text, start_marker, stop_markers=None):
    """
    Extract text starting from start_marker and stopping before the first stop_marker.
    If start_marker is not found, returns empty string.
    """
    if not text:
        return ""

    text = clean_text(text)
    start_idx = text.lower().find(start_marker.lower())
    if start_idx == -1:
        return ""

    chunk = text[start_idx + len(start_marker):]

    if stop_markers:
        stop_positions = []
        lower_chunk = chunk.lower()
        for stop in stop_markers:
            pos = lower_chunk.find(stop.lower())
            if pos != -1:
                stop_positions.append(pos)
        if stop_positions:
            chunk = chunk[:min(stop_positions)]

    return clean_text(chunk)

def keep_first_sentences(text, max_chars=500):
    """
    Optional utility if you want to shorten very long text.
    """
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."

**Convert coursera_skill_catalog.json to CSV**

In [ ]:
coursera_json_path = Path("/content/coursera_skill_catalog.json")
coursera_csv_path = Path("/content/coursera_skill_catalog_flat.csv")

with coursera_json_path.open("r", encoding="utf-8") as f:
    coursera_data = json.load(f)

coursera_rows = []

for item in coursera_data:
    esco_skill_uri = clean_text(item.get("esco_skill_uri"))
    skill_label = clean_text(item.get("skill_label"))
    query = clean_text(item.get("query"))
    result_count = item.get("result_count", 0)

    results = item.get("results", [])
    for idx, res in enumerate(results, start=1):
        coursera_rows.append({
            "esco_skill_uri": esco_skill_uri,
            "skill_label": skill_label,
            "query": query,
            "result_index": idx,
            "course_title": clean_text(res.get("title")),
            "course_url": clean_text(res.get("url")),
            "program_type": clean_text(res.get("program_type")),
            "level": clean_text(res.get("level")),
            "provider": clean_text(res.get("provider")),
            "result_count": result_count
        })

coursera_df = pd.DataFrame(coursera_rows)
coursera_df.to_csv(coursera_csv_path, index=False, encoding="utf-8")

print("Saved:", coursera_csv_path)
print("Rows:", len(coursera_df))
coursera_df.head()

Saved: /content/coursera_skill_catalog_flat.csv
Rows: 1645


,esco_skill_uri,skill_label,query,result_index,course_title,course_url,program_type,level,provider,result_count
0,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell,1,Rust Programming,https://www.coursera.org/specializations/rust-...,specialization,beginner,,5
1,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell,2,Functional Programming in Scala,https://www.coursera.org/specializations/scala,specialization,intermediate,,5
2,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell,3,AI Agents with Model Context Protocol & Typesc...,https://www.coursera.org/specializations/ai-ag...,specialization,beginner,,5
3,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell,4,"C, Go, and C++: A Comprehensive Introduction t...",https://www.coursera.org/specializations/c-go-...,specialization,intermediate,,5
4,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,Haskell,5,Langchain and Langgraph,https://www.coursera.org/specializations/packt...,specialization,intermediate,,5


**Convert google_cloud_certifications_by_role.json to CSV**

In [ ]:
import json
import pandas as pd
from pathlib import Path

gcp_json_path = Path("/content/google_cloud_certifications_by_role.json")
gcp_csv_path = Path("/content/google_cloud_certifications_by_role_flat.csv")

with gcp_json_path.open("r", encoding="utf-8") as f:
    gcp_data = json.load(f)

rows = []

for item in gcp_data:
    custom_role = item.get("custom_role", "")
    certs = item.get("certifications", [])

    for cert in certs:
        rows.append({
            "custom_role": custom_role,
            "cert_key": cert.get("cert_key", ""),
            "cert_name": cert.get("cert_name", ""),
            "tier": cert.get("tier", ""),
            "url": cert.get("url", ""),
            "page_title": cert.get("page_title", ""),
            "meta_description": cert.get("meta_description", "")
        })

gcp_df = pd.DataFrame(rows)
gcp_df.to_csv(gcp_csv_path, index=False, encoding="utf-8")

print("Saved:", gcp_csv_path)
print("Rows:", len(gcp_df))
gcp_df.head()

Saved: /content/google_cloud_certifications_by_role_flat.csv
Rows: 45


,custom_role,cert_key,cert_name,tier,url,page_title,meta_description
0,Software Engineer,associate_cloud_engineer,Associate Cloud Engineer,associate,https://cloud.google.com/learn/certification/c...,Associate Cloud Engineer,"Associate Cloud Engineers deploy apps, monitor..."
1,Software Engineer,professional_cloud_developer,Professional Cloud Developer,professional,https://cloud.google.com/learn/certification/c...,Professional Cloud Developer,Professional Cloud Developers build scalable c...
2,Software Engineer,professional_cloud_architect,Professional Cloud Architect,professional,https://cloud.google.com/learn/certification/c...,Professional Cloud Architect,A Google Certified Professional - Cloud Archit...
3,Backend Engineer,professional_cloud_developer,Professional Cloud Developer,professional,https://cloud.google.com/learn/certification/c...,Professional Cloud Developer,Professional Cloud Developers build scalable c...
4,Backend Engineer,professional_cloud_architect,Professional Cloud Architect,professional,https://cloud.google.com/learn/certification/c...,Professional Cloud Architect,A Google Certified Professional - Cloud Archit...


# Generating **customrole_esco_skill_metrics.csv**

Imports

In [1]:
!pip -q install rapidfuzz pandas

import os
import re
import glob
from collections import defaultdict

import pandas as pd
from rapidfuzz import process, fuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.1 MB/s eta 0:00:00


Paths

In [2]:
ROLE_MAP_PATH = "/content/custom_role_esco_map.csv"
ESCO_REQ_PATH = "/content/final_esco_occ_requires_skill_tech.csv"
ESCO_SKILLS_PATH = "/content/final_esco_skills_tech.csv"
COUNTS_DIR = "/content/LinkedIn_top_1000_skills_freq_by_CR_code"
OUT_PATH = "/content/customrole_esco_skill_metrics.csv"

Helper functions

In [3]:
def clean_text(x):
    if pd.isna(x):
        return ""
    text = str(x).replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(x):
    """
    Normalization that preserves + and # so C++, C#, .NET-like skills don't break too badly.
    """
    text = clean_text(x).lower()
    text = re.sub(r"\(.*?\)", " ", text)           # remove parenthetical text
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9+#\s]", " ", text)     # keep letters, numbers, +, #
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_aliases(text):
    """
    Split skill alternative labels safely.
    """
    if pd.isna(text) or not str(text).strip():
        return []
    raw = str(text)
    parts = re.split(r"[\n;]+", raw)
    return [clean_text(p) for p in parts if clean_text(p)]

def detect_col(df, candidates, required=True):
    cols = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    if required:
        raise ValueError(f"Could not find any of these columns: {candidates}. Available: {list(df.columns)}")
    return None

def mode_or_first(series):
    s = series.dropna().astype(str)
    if s.empty:
        return ""
    modes = s.mode()
    return modes.iloc[0] if not modes.empty else s.iloc[0]

def get_role_count_file(role_id):
    """
    role_id like CR01 -> 01_skill_counts.csv
    """
    m = re.search(r"(\d+)", str(role_id))
    if not m:
        return None
    num = int(m.group(1))
    candidates = [
        os.path.join(COUNTS_DIR, f"{num:02d}_skill_counts.csv"),
        os.path.join(COUNTS_DIR, f"{num}_skill_counts.csv"),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path

    # fallback glob
    hits = glob.glob(os.path.join(COUNTS_DIR, f"*{num:02d}*skill_counts.csv"))
    if hits:
        return hits[0]
    hits = glob.glob(os.path.join(COUNTS_DIR, f"*{num}*skill_counts.csv"))
    return hits[0] if hits else None

Loading Inputs

In [7]:
role_map = pd.read_csv(ROLE_MAP_PATH)
custom_roles = pd.read_csv("/content/custom_roles.csv")
occ_req = pd.read_csv(ESCO_REQ_PATH)
skill_df = pd.read_csv(ESCO_SKILLS_PATH)

role_map.columns = [c.strip() for c in role_map.columns]
occ_req.columns = [c.strip() for c in occ_req.columns]
skill_df.columns = [c.strip() for c in skill_df.columns]

custom_roles.columns = [c.strip() for c in custom_roles.columns]
custom_roles["role_id"] = custom_roles["role_id"].astype(str).str.strip()
custom_roles["role_title"] = custom_roles["role_title"].astype(str).str.strip()

role_map = role_map.merge(
    custom_roles[["role_id", "role_title"]],
    on="role_id",
    how="left"
)

role_conf_col = detect_col(role_map, ["mapping_confidence", "confidence", "match_confidence"], required=False)
if role_conf_col is None:
    role_map["mapping_confidence"] = 1.0
    role_conf_col = "mapping_confidence"
else:
    role_map = role_map.rename(columns={role_conf_col: "mapping_confidence"})
    role_conf_col = "mapping_confidence"

# Required columns in role_map
role_id_col = detect_col(role_map, ["role_id"])
role_title_col = detect_col(role_map, ["role_title"])
role_esco_uri_col = detect_col(role_map, ["esco_occ_uri"])
role_esco_label_col = detect_col(role_map, ["esco_occ_label"])

# Required columns in occ_req
occ_uri_col = detect_col(occ_req, ["esco_occ_uri"])
skill_uri_col_req = detect_col(occ_req, ["esco_skill_uri"])
relation_col = detect_col(occ_req, ["relation_type"])
skill_type_col = detect_col(occ_req, ["skill_type_in_occ"])

# Required columns in skill_df
skill_uri_col = detect_col(skill_df, ["esco_skill_uri"])
skill_label_col = detect_col(skill_df, ["label", "skill_label", "skill_preferred_label"])
skill_alt_col = detect_col(skill_df, ["skill_alternative_label"], required=False)

# Clean string columns
for col in [role_id_col, role_title_col, role_esco_uri_col, role_esco_label_col, role_conf_col]:
    role_map[col] = role_map[col].apply(clean_text)

for col in [occ_uri_col, skill_uri_col_req, relation_col, skill_type_col]:
    occ_req[col] = occ_req[col].apply(clean_text)

skill_df[skill_uri_col] = skill_df[skill_uri_col].apply(clean_text)
skill_df[skill_label_col] = skill_df[skill_label_col].apply(clean_text)
if skill_alt_col:
    skill_df[skill_alt_col] = skill_df[skill_alt_col].apply(clean_text)

# Remove duplicate role-occupation mappings if any
role_map = role_map.drop_duplicates(subset=[role_id_col, role_esco_uri_col]).copy()

Build ESCO skill alias lookup for fuzzy matching

In [8]:
alias_to_skills = defaultdict(list)
alias_strings = []

for _, row in skill_df.iterrows():
    uri = row[skill_uri_col]
    canonical = row[skill_label_col]
    aliases = [canonical]
    if skill_alt_col:
        aliases.extend(split_aliases(row[skill_alt_col]))

    for a in aliases:
        a_norm = normalize_text(a)
        if not a_norm:
            continue
        alias_to_skills[a_norm].append({
            "esco_skill_uri": uri,
            "skill_label": canonical,
            "alias": a
        })

alias_strings = list(alias_to_skills.keys())

def match_linkedin_skill_to_esco(linkedin_skill_label):
    """
    Returns:
      (esco_skill_uri, skill_label, match_score, matched_alias)
    or None
    """
    q = normalize_text(linkedin_skill_label)
    if not q:
        return None

    # exact normalized match first
    if q in alias_to_skills:
        candidates = alias_to_skills[q]
        # Prefer exact canonical label match if present
        for cand in candidates:
            if normalize_text(cand["skill_label"]) == q:
                return cand["esco_skill_uri"], cand["skill_label"], 100, q
        best = candidates[0]
        return best["esco_skill_uri"], best["skill_label"], 100, q

    # fuzzy match fallback
    if len(q) <= 2:
        return None

    threshold = 95 if len(q) <= 4 else 88
    best = process.extractOne(q, alias_strings, scorer=fuzz.WRatio)
    if best and best[1] >= threshold:
        matched_alias = best[0]
        cand = alias_to_skills[matched_alias][0]
        return cand["esco_skill_uri"], cand["skill_label"], best[1], matched_alias

    return None

Helper function for manual parsing of xx_skill_counts.csv file

In [11]:
def read_skill_count_file(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            # skip header
            if i == 1 and "skill" in line.lower() and "count" in line.lower():
                continue

            # skip summary row
            if line.lower().startswith("total_count"):
                continue

            parts = line.rsplit(",", 1)
            if len(parts) != 2:
                continue

            skill, count = parts[0].strip(), parts[1].strip()
            try:
                count = int(float(count))
            except:
                continue

            rows.append({"skill": skill, "count": count})

    return pd.DataFrame(rows)

Build metrics file rows by role

In [12]:
all_rows = []
role_ids = role_map[role_id_col].dropna().unique().tolist()

for role_id in sorted(role_ids):
    role_rows = role_map[role_map[role_id_col] == role_id].copy()
    if role_rows.empty:
        continue

    role_title = role_rows[role_title_col].iloc[0]
    mapped_occ_uris = role_rows[role_esco_uri_col].dropna().unique().tolist()
    total_mapped_occ = len(mapped_occ_uris)

    if total_mapped_occ == 0:
        print(f"[WARN] No ESCO occupations found for {role_id} ({role_title})")
        continue

    # role_id -> count file
    count_file = get_role_count_file(role_id)
    linkedin_count_map = {}

    if count_file and os.path.exists(count_file):
        count_df = read_skill_count_file(count_file)
        if count_df.empty:
          print(f"[WARN] {role_id}: no valid rows in {os.path.basename(count_file)}")
          continue

        count_skill_col = "skill"
        count_val_col = "count"

        count_df[count_skill_col] = count_df[count_skill_col].apply(clean_text)
        count_df[count_val_col] = pd.to_numeric(count_df[count_val_col], errors="coerce").fillna(0).astype(int)

        # map each LinkedIn skill label to an ESCO skill URI
        for _, crow in count_df.iterrows():
            li_skill = crow[count_skill_col]
            li_count = int(crow[count_val_col])

            match = match_linkedin_skill_to_esco(li_skill)
            if not match:
                continue

            esco_skill_uri, esco_skill_label, match_score, matched_alias = match

            # keep the highest observed count for this ESCO skill
            prev = linkedin_count_map.get(esco_skill_uri, 0)
            if li_count > prev:
                linkedin_count_map[esco_skill_uri] = li_count

        print(f"[OK] {role_id} -> loaded {os.path.basename(count_file)} | matched ESCO skills: {len(linkedin_count_map)}")
    else:
        print(f"[WARN] No LinkedIn count file found for {role_id}; counts will be 0.")

    # Expand role -> occupation -> skills
    role_occ_skill = occ_req[occ_req[occ_uri_col].isin(mapped_occ_uris)].copy()
    if role_occ_skill.empty:
        print(f"[WARN] No occupation-skill rows found for {role_id}")
        continue

    role_occ_skill = role_occ_skill.merge(
        role_rows[[role_esco_uri_col, role_esco_label_col, role_conf_col]],
        left_on=occ_uri_col,
        right_on=role_esco_uri_col,
        how="left"
    )

    role_occ_skill = role_occ_skill.merge(
        skill_df[[skill_uri_col, skill_label_col] + ([skill_alt_col] if skill_alt_col else [])],
        left_on=skill_uri_col_req,
        right_on=skill_uri_col,
        how="left"
    )

    # Aggregate to one row per unique ESCO skill for the role
    grouped = []
    for skill_uri, sub in role_occ_skill.groupby(skill_uri_col_req):
        sub = sub.copy()

        # primary supporting occupation = highest confidence, then essential over optional
        rel_priority = sub[relation_col].str.lower().map({"essential": 0, "optional": 1}).fillna(1)
        sub["_rel_priority"] = rel_priority
        sub["_mapping_conf"] = pd.to_numeric(sub[role_conf_col], errors="coerce").fillna(0.0)

        sub = sub.sort_values(by=["_mapping_conf", "_rel_priority"], ascending=[False, True])
        primary = sub.iloc[0]

        relation_type = "essential" if sub[relation_col].astype(str).str.lower().eq("essential").any() else "optional"
        skill_type_in_occ = mode_or_first(sub[skill_type_col])

        supporting_occ_count = sub[occ_uri_col].nunique()
        skill_overlap_value = supporting_occ_count / total_mapped_occ if total_mapped_occ else 0.0

        linkedin_skill_count = int(linkedin_count_map.get(skill_uri, 0))
        skill_label = clean_text(primary.get(skill_label_col, "")) or skill_uri

        grouped.append({
            "role_id": role_id,
            "role_title": role_title,
            "esco_occ_uri": clean_text(primary[role_esco_uri_col]),
            "esco_occ_label": clean_text(primary[role_esco_label_col]),
            "esco_skill_uri": clean_text(skill_uri),
            "skill_label": skill_label,
            "relation_type": relation_type,
            "skill_type_in_occ": skill_type_in_occ,
            "mapping_confidence": float(primary[role_conf_col]) if pd.notna(primary[role_conf_col]) else 0.0,
            "skill_overlap_count": int(supporting_occ_count),
            "skill_overlap_value": float(skill_overlap_value),
            "linkedin_skill_count": linkedin_skill_count,
        })

    agg = pd.DataFrame(grouped)

    if agg.empty:
        print(f"[WARN] No aggregated skills for {role_id}")
        continue

    # Normalize LinkedIn counts within role
    max_count = agg["linkedin_skill_count"].max()

    if max_count > 0:
        agg["skill_importance"] = agg["linkedin_skill_count"] / max_count
    else:
        # fallback if no LinkedIn match exists
        agg["skill_importance"] = agg["skill_overlap_value"]

    # Rank with deterministic tie-breaks
    agg["relation_priority"] = agg["relation_type"].str.lower().map({"essential": 0, "optional": 1}).fillna(1)
    agg = agg.sort_values(
        by=["skill_importance", "skill_overlap_value", "relation_priority", "mapping_confidence", "skill_label"],
        ascending=[False, False, True, False, True]
    ).reset_index(drop=True)

    agg["skill_rank"] = range(1, len(agg) + 1)
    agg["is_top20"] = agg["skill_rank"].apply(lambda x: "Y" if x <= 20 else "N")

    # Keep only the requested columns (plus overlap value + LinkedIn count for transparency)
    agg = agg[[
        "role_id",
        "role_title",
        "esco_occ_uri",
        "esco_occ_label",
        "esco_skill_uri",
        "skill_label",
        "relation_type",
        "skill_type_in_occ",
        "mapping_confidence",
        "skill_overlap_count",
        "skill_overlap_value",
        "linkedin_skill_count",
        "skill_importance",
        "skill_rank",
        "is_top20"
    ]]

    all_rows.append(agg)

    print(f"[DONE] {role_id} | unique skills: {len(agg)} | top20: {(agg['is_top20'] == 'Y').sum()}")

[OK] CR01 -> loaded 01_skill_counts.csv | matched ESCO skills: 184
[DONE] CR01 | unique skills: 140 | top20: 20
[OK] CR02 -> loaded 02_skill_counts.csv | matched ESCO skills: 135
[DONE] CR02 | unique skills: 155 | top20: 20
[OK] CR03 -> loaded 03_skill_counts.csv | matched ESCO skills: 177
[DONE] CR03 | unique skills: 137 | top20: 20
[OK] CR04 -> loaded 04_skill_counts.csv | matched ESCO skills: 146
[DONE] CR04 | unique skills: 139 | top20: 20
[OK] CR05 -> loaded 05_skill_counts.csv | matched ESCO skills: 181
[DONE] CR05 | unique skills: 180 | top20: 20
[OK] CR06 -> loaded 06_skill_counts.csv | matched ESCO skills: 155
[DONE] CR06 | unique skills: 175 | top20: 20
[OK] CR07 -> loaded 07_skill_counts.csv | matched ESCO skills: 148
[DONE] CR07 | unique skills: 175 | top20: 20
[OK] CR08 -> loaded 08_skill_counts.csv | matched ESCO skills: 170
[DONE] CR08 | unique skills: 168 | top20: 20
[OK] CR09 -> loaded 09_skill_counts.csv | matched ESCO skills: 156
[DONE] CR09 | unique skills: 156 | to

Saving output

In [13]:
final_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# Optional: round numeric columns for readability
for col in ["mapping_confidence", "skill_overlap_value", "skill_importance"]:
    if col in final_df.columns:
        final_df[col] = final_df[col].round(4)

final_df.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("\nSaved:", OUT_PATH)
print("Total rows:", len(final_df))

# Preview
final_df.head(20)


Saved: /content/customrole_esco_skill_metrics.csv
Total rows: 2404


,role_id,role_title,esco_occ_uri,esco_occ_label,esco_skill_uri,skill_label,relation_type,skill_type_in_occ,mapping_confidence,skill_overlap_count,skill_overlap_value,linkedin_skill_count,skill_importance,skill_rank,is_top20
0,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/19a8293b-8e95...,Java (computer programming),optional,knowledge,0.97,2,1.0,4606,1.0000,1,Y
1,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/ccd0a1d9-afda...,Python (computer programming),optional,knowledge,0.97,2,1.0,4587,0.9959,2,Y
2,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/598de5b0-5b58...,SQL,optional,knowledge,0.97,1,0.5,3068,0.6661,3,Y
3,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/172020d1-e151...,utilise computer-aided software engineering tools,essential,skill/competence,0.97,2,1.0,2893,0.6281,4,Y
4,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/51586df8-1c46...,R,optional,knowledge,0.97,2,1.0,2550,0.5536,5,Y
5,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/3cd569a2-4f88...,JavaScript,optional,knowledge,0.97,2,1.0,2435,0.5287,6,Y
6,CR01,Software Engineer,http://data.europa.eu/esco/occupation/d0aa0792...,software architect,http://data.europa.eu/esco/skill/0a9acb6b-1139...,Agile project management,optional,knowledge,0.78,1,0.5,2341,0.5083,7,Y
7,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/b633eb55-8f1f...,C++,optional,knowledge,0.97,2,1.0,2140,0.4646,8,Y
8,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/04f1b938-d4d4...,SAS language,optional,knowledge,0.97,2,1.0,1977,0.4292,9,Y
9,CR01,Software Engineer,http://data.europa.eu/esco/occupation/f2b15a0e...,software developer,http://data.europa.eu/esco/skill/4c016b68-4116...,C#,optional,knowledge,0.97,2,1.0,1365,0.2964,10,Y


# Generating **customrole_skill_thresholds.csv**

In [14]:
import pandas as pd
import re
from pathlib import Path

INPUT_PATH = Path("/content/customrole_esco_skill_metrics.csv")
OUTPUT_PATH = Path("/content/customrole_skill_thresholds.csv")

df = pd.read_csv(INPUT_PATH)

Helper functions

In [15]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def clamp(n, low=1, high=5):
    return max(low, min(high, int(n)))

def to_bool_top20(x):
    s = str(x).strip().lower()
    return s in {"y", "yes", "1", "true", "t"}

def compute_thresholds(skill_importance, relation_type):
    """
    Bands:
      >= 0.85 -> min 4, fav 5
      >= 0.70 -> min 3, fav 4
      >= 0.50 -> min 2, fav 3
      <  0.50 -> min 1, fav 2

    Then:
      essential -> minimum_required + 1 (capped at 5)
    """
    if skill_importance >= 0.85:
        minimum_required = 4
        favorable_target = 5
        band_label = "high"
    elif skill_importance >= 0.70:
        minimum_required = 3
        favorable_target = 4
        band_label = "medium-high"
    elif skill_importance >= 0.50:
        minimum_required = 2
        favorable_target = 3
        band_label = "medium"
    else:
        minimum_required = 1
        favorable_target = 2
        band_label = "low"

    relation_type = clean_text(relation_type).lower()
    if relation_type == "essential":
        minimum_required = clamp(minimum_required + 1, 1, 5)

    # Ensure favorable is always above minimum when possible
    favorable_target = clamp(favorable_target, 1, 5)
    if favorable_target <= minimum_required:
        favorable_target = clamp(minimum_required + 1, 1, 5)

    return minimum_required, favorable_target, band_label

def build_reason(skill_label, relation_type, skill_importance, minimum_required, favorable_target, band_label):
    relation_type = clean_text(relation_type).lower()
    if relation_type == "essential":
        core_text = "core skill"
    else:
        core_text = "supporting skill"

    if band_label == "high":
        demand_text = "highly important and frequently demanded"
    elif band_label == "medium-high":
        demand_text = "important and commonly expected"
    elif band_label == "medium":
        demand_text = "useful and moderately important"
    else:
        demand_text = "helpful but not central"

    return (
        f"{skill_label} is a {core_text} for this role, "
        f"{demand_text} based on the ranking signal. "
        f"Minimum acceptable proficiency is {minimum_required}/5, "
        f"and favorable proficiency is {favorable_target}/5."
    )

**Keeping only top 20 skills per role**

In [16]:
if "is_top20" in df.columns:
    top_df = df[df["is_top20"].apply(to_bool_top20)].copy()
else:
    top_df = df.copy()

# Make sure we have the needed columns
needed = ["role_id", "role_title", "esco_skill_uri", "skill_label", "relation_type", "skill_importance"]
missing = [c for c in needed if c not in top_df.columns]
if missing:
    raise ValueError(f"Missing required columns in metrics file: {missing}")

**Generating thresholds**

In [17]:
rows = []

for _, row in top_df.iterrows():
    role_id = clean_text(row["role_id"])
    role_title = clean_text(row["role_title"])
    esco_skill_uri = clean_text(row["esco_skill_uri"])
    skill_label = clean_text(row["skill_label"])
    relation_type = clean_text(row.get("relation_type", "optional"))
    skill_importance = float(row.get("skill_importance", 0) or 0)

    minimum_required, favorable_target, band_label = compute_thresholds(
        skill_importance=skill_importance,
        relation_type=relation_type
    )

    threshold_reason = build_reason(
        skill_label=skill_label,
        relation_type=relation_type,
        skill_importance=skill_importance,
        minimum_required=minimum_required,
        favorable_target=favorable_target,
        band_label=band_label
    )

    rows.append({
        "role_id": role_id,
        "role_title": role_title,
        "esco_skill_uri": esco_skill_uri,
        "skill_label": skill_label,
        "relation_type": relation_type,
        "skill_importance": round(skill_importance, 4),
        "minimum_required": minimum_required,
        "favorable_target": favorable_target,
        "threshold_reason": threshold_reason
    })

threshold_df = pd.DataFrame(rows)

# Optional: sort for readability
threshold_df = threshold_df.sort_values(
    by=["role_id", "skill_importance", "skill_label"],
    ascending=[True, False, True]
).reset_index(drop=True)

Saving to output file

In [18]:
threshold_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(threshold_df)}")
threshold_df.head(20)

Saved: /content/customrole_skill_thresholds.csv
Rows: 300


,role_id,role_title,esco_skill_uri,skill_label,relation_type,skill_importance,minimum_required,favorable_target,threshold_reason
0,CR01,Software Engineer,http://data.europa.eu/esco/skill/19a8293b-8e95...,Java (computer programming),optional,1.0000,4,5,Java (computer programming) is a supporting sk...
1,CR01,Software Engineer,http://data.europa.eu/esco/skill/ccd0a1d9-afda...,Python (computer programming),optional,0.9959,4,5,Python (computer programming) is a supporting ...
2,CR01,Software Engineer,http://data.europa.eu/esco/skill/598de5b0-5b58...,SQL,optional,0.6661,2,3,"SQL is a supporting skill for this role, usefu..."
3,CR01,Software Engineer,http://data.europa.eu/esco/skill/172020d1-e151...,utilise computer-aided software engineering tools,essential,0.6281,3,4,utilise computer-aided software engineering to...
4,CR01,Software Engineer,http://data.europa.eu/esco/skill/51586df8-1c46...,R,optional,0.5536,2,3,"R is a supporting skill for this role, useful ..."
5,CR01,Software Engineer,http://data.europa.eu/esco/skill/3cd569a2-4f88...,JavaScript,optional,0.5287,2,3,JavaScript is a supporting skill for this role...
6,CR01,Software Engineer,http://data.europa.eu/esco/skill/0a9acb6b-1139...,Agile project management,optional,0.5083,2,3,Agile project management is a supporting skill...
7,CR01,Software Engineer,http://data.europa.eu/esco/skill/b633eb55-8f1f...,C++,optional,0.4646,1,2,"C++ is a supporting skill for this role, helpf..."
8,CR01,Software Engineer,http://data.europa.eu/esco/skill/04f1b938-d4d4...,SAS language,optional,0.4292,1,2,SAS language is a supporting skill for this ro...
9,CR01,Software Engineer,http://data.europa.eu/esco/skill/4c016b68-4116...,C#,optional,0.2964,1,2,"C# is a supporting skill for this role, helpfu..."
